# Agent Position Log Analysis
Analyzes `position_log.csv` produced by `OverseerCleanupEnv` during training.
Logs one in every `log_every_n_episodes` episodes, full step-by-step trajectory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "results/position_log.csv"  # adjust if needed
GRID_SIZE = 8  # match your env's `size`

df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## Fix column misalignment
If the CSV has 5 raw columns but only 4 header names, pandas silently shifts everything
left by one -- the leftmost column is actually a stray row index, not `episode`.
This cell detects and fixes that automatically.

In [ ]:
if list(df.columns) == ["episode", "step", "x", "y"] and df["y"].isna().any() is False and df.shape[1] == 4:
    # check if there's a leading unnamed index-like column by re-reading raw
    raw = pd.read_csv(CSV_PATH, header=None)
    n_data_cols = raw.shape[1]
    if n_data_cols == 5:
        # first row is the header; re-load skipping it, assign 5 names, drop stray index col
        raw = pd.read_csv(CSV_PATH, header=0, names=["row_idx", "episode", "step", "x", "y"])
        df = raw.drop(columns=["row_idx"])
print(df.dtypes)
df.head()

## Which episodes were logged?

In [ ]:
logged_episodes = sorted(df["episode"].unique())
print(f"{len(logged_episodes)} episodes logged: {logged_episodes}")

## Visitation heatmap per logged episode
Shows where the agent spent time within each sampled episode -- corners/edges parked in
for many steps suggest idling; broad even coverage suggests active exploration/chasing.

In [ ]:
def episode_heatmap(ep_df, grid_size=GRID_SIZE):
    heat = np.zeros((grid_size, grid_size))
    for x, y in zip(ep_df["x"], ep_df["y"]):
        if 0 <= x < grid_size and 0 <= y < grid_size:
            heat[y, x] += 1
    return heat

n = len(logged_episodes)
ncols = min(4, n)
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = np.array(axes).reshape(-1)

for i, ep in enumerate(logged_episodes):
    ep_df = df[df["episode"] == ep]
    heat = episode_heatmap(ep_df)
    im = axes[i].imshow(heat, cmap="hot", origin="upper")
    axes[i].set_title(f"episode {ep}")
    fig.colorbar(im, ax=axes[i], fraction=0.046)

for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig("position_heatmaps.png", dpi=120)
plt.show()

## Trajectory path for a single episode
Line plot of movement order -- useful to see loops/back-and-forth vs directed exploration.

In [ ]:
EPISODE_TO_PLOT = logged_episodes[0]  # change to inspect a different one

ep_df = df[df["episode"] == EPISODE_TO_PLOT].sort_values("step")
plt.figure(figsize=(6, 6))
plt.plot(ep_df["x"], ep_df["y"], marker="o", markersize=2, linewidth=0.8, alpha=0.7)
plt.scatter(ep_df["x"].iloc[0], ep_df["y"].iloc[0], color="green", s=80, label="start", zorder=5)
plt.scatter(ep_df["x"].iloc[-1], ep_df["y"].iloc[-1], color="red", s=80, label="end", zorder=5)
plt.xlim(-0.5, GRID_SIZE - 0.5)
plt.ylim(GRID_SIZE - 0.5, -0.5)  # invert y to match grid row orientation
plt.gca().set_aspect("equal")
plt.title(f"Trajectory -- episode {EPISODE_TO_PLOT}")
plt.legend()
plt.savefig(f"trajectory_ep{EPISODE_TO_PLOT}.png", dpi=120)
plt.show()

## Movement statistics per episode
- `unique_cells_visited`: coverage breadth
- `revisit_ratio`: fraction of steps spent on already-visited cells (high = looping/idling)
- `edge_time_ratio` / `corner_time_ratio`: time spent hugging boundaries (possible idling signature)

In [ ]:
def episode_stats(ep_df, grid_size=GRID_SIZE):
    coords = list(zip(ep_df["x"], ep_df["y"]))
    total_steps = len(coords)
    unique_cells = len(set(coords))

    seen = set()
    revisits = 0
    for c in coords:
        if c in seen:
            revisits += 1
        seen.add(c)

    edge_steps = sum(1 for x, y in coords if x in (0, grid_size - 1) or y in (0, grid_size - 1))
    corner_steps = sum(1 for x, y in coords
                        if x in (0, grid_size - 1) and y in (0, grid_size - 1))

    return {
        "total_steps": total_steps,
        "unique_cells_visited": unique_cells,
        "coverage_pct": unique_cells / (grid_size * grid_size),
        "revisit_ratio": revisits / total_steps,
        "edge_time_ratio": edge_steps / total_steps,
        "corner_time_ratio": corner_steps / total_steps,
    }

stats_rows = []
for ep in logged_episodes:
    ep_df = df[df["episode"] == ep].sort_values("step")
    row = {"episode": ep, **episode_stats(ep_df)}
    stats_rows.append(row)

stats_df = pd.DataFrame(stats_rows)
stats_df

## Stats trend across training
Plots coverage/revisit/edge-time ratios against episode number -- shows whether
exploration behavior changes as training progresses (e.g. more idling later on
if the agent has exhausted local dirt and hasn't learned to move efficiently).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(stats_df["episode"], stats_df["coverage_pct"], marker="o")
axes[0].set_title("Grid coverage %")
axes[0].set_xlabel("episode")

axes[1].plot(stats_df["episode"], stats_df["revisit_ratio"], marker="o", color="orange")
axes[1].set_title("Revisit ratio")
axes[1].set_xlabel("episode")

axes[2].plot(stats_df["episode"], stats_df["edge_time_ratio"], marker="o", color="green", label="edge")
axes[2].plot(stats_df["episode"], stats_df["corner_time_ratio"], marker="o", color="red", label="corner")
axes[2].set_title("Boundary time ratio")
axes[2].set_xlabel("episode")
axes[2].legend()

plt.tight_layout()
plt.savefig("movement_stats_trend.png", dpi=120)
plt.show()

## Notes / how to read this
- **Low coverage_pct + high revisit_ratio** in an episode -> agent is looping/idling in a
  small region, consistent with the "nothing left to do, wasting steps" hypothesis.
- **High coverage_pct, low revisit_ratio** -> agent is actively roaming, likely still
  chasing/relocating dirt across the grid.
- **Rising edge/corner time in later episodes** -> possible sign the agent retreats to a
  boundary once local dirt is exhausted (a cheap way to minimize movement cost while
  waiting out the rest of the episode) -- worth checking against `wasted_actions` from
  the training info dict for the same episodes, if available.